In [31]:
!pip install prophet statsmodels plotly scikit-learn nbformat

In [32]:
import os

base = "pk_it_demand_dashboard"

folders = [
    "01_raw_data",        # scraped / downloaded files land here
    "02_clean_data",      # processed CSVs Power BI will read
    "03_forecasts",       # Prophet / statsmodels output CSVs
    "04_notebooks",       # Jupyter notebooks
    "05_powerbi",         # .pbix file lives here
    "06_exports",         # PDF exports, screenshots for LinkedIn
]

for folder in folders:
    path = os.path.join(base, folder)
    os.makedirs(path, exist_ok=True)
    print(f"✅ Created: {path}")

print("\n📁 Project scaffold ready!")

✅ Created: pk_it_demand_dashboard\01_raw_data
✅ Created: pk_it_demand_dashboard\02_clean_data
✅ Created: pk_it_demand_dashboard\03_forecasts
✅ Created: pk_it_demand_dashboard\04_notebooks
✅ Created: pk_it_demand_dashboard\05_powerbi
✅ Created: pk_it_demand_dashboard\06_exports

📁 Project scaffold ready!


In [60]:
import os

# ── Hardcoded absolute project root — bulletproof regardless of Jupyter CWD ──
PROJECT_ROOT = r"C:\Users\User\pk_it_demand_dashboard"
RAW       = os.path.join(PROJECT_ROOT, "01_raw_data")
CLEAN     = os.path.join(PROJECT_ROOT, "02_clean_data")
FORECASTS = os.path.join(PROJECT_ROOT, "03_forecasts")

# Create all folders if missing
for folder in [RAW, CLEAN, FORECASTS]:
    os.makedirs(folder, exist_ok=True)

print(f"✅ PROJECT_ROOT : {PROJECT_ROOT}")
print(f"✅ CLEAN        : {CLEAN}")
print(f"✅ Folder exists: {os.path.isdir(CLEAN)}")

✅ PROJECT_ROOT : C:\Users\User\pk_it_demand_dashboard
✅ CLEAN        : C:\Users\User\pk_it_demand_dashboard\02_clean_data
✅ Folder exists: True


In [33]:
# ============================================================
# CELL 1 — Imports & config (run this first every session)
# ============================================================
import os
import time
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore")

# ── Paths (relative to notebook location) ──
ROOT       = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW        = os.path.join(ROOT, "01_raw_data")
CLEAN      = os.path.join(ROOT, "02_clean_data")
FORECASTS  = os.path.join(ROOT, "03_forecasts")

# ── Headers for scraping (polite browser identity) ──
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

print("✅ Imports done")
print(f"   RAW      → {RAW}")
print(f"   CLEAN    → {CLEAN}")
print(f"   FORECAST → {FORECASTS}")

✅ Imports done
   RAW      → C:\Users\01_raw_data
   CLEAN    → C:\Users\02_clean_data
   FORECAST → C:\Users\03_forecasts


In [34]:
# CELL 2 — Dependency check
import importlib

required = {
    "pandas":       "pd",
    "numpy":        "np",
    "requests":     "requests",
    "bs4":          "BeautifulSoup",
    "prophet":      "Prophet",
    "statsmodels":  "statsmodels",
    "plotly":       "plotly",
    "sklearn":      "scikit-learn",
    "openpyxl":     "openpyxl",
}

all_good = True
for module, label in required.items():
    try:
        importlib.import_module(module)
        print(f"  ✅  {label}")
    except ImportError:
        print(f"  ❌  {label}  ← run: pip install {module}")
        all_good = False

print()
print("🚀 Ready for Phase 2!" if all_good else "⚠️  Fix the ❌ items above first.")

  ✅  pd
  ✅  np
  ✅  requests
  ✅  BeautifulSoup
  ✅  Prophet
  ✅  statsmodels
  ✅  plotly
  ✅  scikit-learn
  ✅  openpyxl

🚀 Ready for Phase 2!


# Phase 2

In [36]:
# Source 1: World Bank API (live fetch)
# ============================================================
# CELL 3 — World Bank API: GDP, internet users, ICT exports
# ============================================================
import requests, pandas as pd, os, time

CLEAN = "C:\\Users\\User\\pk_it_demand_dashboard\\02_clean_data"
os.makedirs(CLEAN, exist_ok=True)

WB_BASE = "https://api.worldbank.org/v2/country/PK/indicator"

indicators = {
    "NY.GDP.MKTP.CD":   "gdp_usd",            # GDP current USD
    "IT.NET.USER.ZS":   "internet_users_pct",  # Internet users % of pop
    "BX.GSR.CCIS.ZS":   "ict_services_exports_pct_service_exports",
    "SP.POP.TOTL":       "population",
}

def fetch_wb(indicator_code, label):
    url = (f"{WB_BASE}/{indicator_code}"
           f"?format=json&per_page=100&mrv=25")
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    raw = r.json()
    
    if len(raw) < 2 or not raw[1]:
        print(f"  ⚠️  No data for {label}")
        return pd.DataFrame()
    
    rows = [
        {"year": int(rec["date"]), label: rec["value"]}
        for rec in raw[1]
        if rec["value"] is not None
    ]
    df = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    print(f"  ✅  {label}: {len(df)} rows ({df.year.min()}–{df.year.max()})")
    return df

# Fetch all indicators and merge into one DataFrame
print("📡 Fetching World Bank data...")
merged = None
for code, label in indicators.items():
    df = fetch_wb(code, label)
    time.sleep(0.5)  # polite delay
    if df.empty:
        continue
    merged = df if merged is None else merged.merge(df, on="year", how="outer")

merged = merged[merged["year"] >= 2000].sort_values("year").reset_index(drop=True)

# GDP in billions for readability
merged["gdp_usd_bn"] = (merged["gdp_usd"] / 1e9).round(2)

out = os.path.join(CLEAN, "wb_macro_indicators.csv")
merged.to_csv(out, index=False)
print(f"\n✅ Saved: {out}")
print(merged.tail())

📡 Fetching World Bank data...
  ✅  gdp_usd: 25 rows (2000–2024)
  ✅  internet_users_pct: 25 rows (1996–2024)
  ✅  ict_services_exports_pct_service_exports: 25 rows (2000–2024)
  ✅  population: 25 rows (2000–2024)

✅ Saved: C:\Users\User\pk_it_demand_dashboard\02_clean_data\wb_macro_indicators.csv
    year       gdp_usd  internet_users_pct  \
20  2020  3.004256e+11              18.935   
21  2021  3.485166e+11                 NaN   
22  2022  3.748903e+11                 NaN   
23  2023  3.366863e+11                 NaN   
24  2024  3.715700e+11              57.253   

    ict_services_exports_pct_service_exports   population  gdp_usd_bn  
20                                 31.824513  235001746.0      300.43  
21                                 37.454042  239477801.0      348.52  
22                                 35.362755  243700667.0      374.89  
23                                 36.702214  247504495.0      336.69  
24                                 44.841978  251269164.0      37

In [37]:
df = pd.read_csv("C:\\Users\\User\\pk_it_demand_dashboard\\02_clean_data\\wb_macro_indicators.csv")
df

,year,gdp_usd,internet_users_pct,ict_services_exports_pct_service_exports,population,gdp_usd_bn
0,2000,9.948480e+10,NaN,15.820896,154879127.0,99.48
1,2001,9.714562e+10,1.318551,15.585331,159270907.0,97.15
2,2002,9.792330e+10,2.577427,14.800839,163222549.0,97.92
3,2003,1.123719e+11,5.041158,7.587615,167110248.0,112.37
4,2004,1.322160e+11,6.164321,9.868864,171286000.0,132.22
5,2005,1.452086e+11,6.332300,9.361354,175453212.0,145.21
6,2006,1.618714e+11,6.500000,6.992806,179682690.0,161.87
7,2007,1.841409e+11,6.800000,6.759657,184493231.0,184.14
8,2008,2.022037e+11,7.000000,6.519181,189499113.0,202.20
9,2009,1.873378e+11,7.500000,11.776598,194376534.0,187.34


In [38]:
#Source 2: SBP IT Export Series (FY2006–FY2025)
# ============================================================
# CELL 4 — SBP IT Exports (Telecom, Computer & Info Services)
# Source: SBP Statistical Bulletin July 2025, Table verified
# https://www.sbp.org.pk/reports/stat_reviews/Bulletin/2025/Jul/
# ============================================================

sbp_data = {
    "fiscal_year":    ["FY06","FY07","FY08","FY09","FY10",
                       "FY11","FY12","FY13","FY14","FY15",
                       "FY16","FY17","FY18","FY19","FY20",
                       "FY21","FY22","FY23","FY24","FY25"],
    "it_exports_mn_usd": [269, 227, 269, 380, 433,
                          443, 460, 803, 817, 821,
                          789, 940, 1067, 1192, 1440,
                          2108, 2619, 2596, 3223, 3809],
}

df_sbp = pd.DataFrame(sbp_data)

# Add calendar year (FY06 ends June 2006 → year 2006)
df_sbp["year"] = [int("20" + fy[2:]) for fy in df_sbp["fiscal_year"]]

# YoY growth
df_sbp["it_exports_yoy_pct"] = (
    df_sbp["it_exports_mn_usd"]
    .pct_change()
    .mul(100)
    .round(1)
)

# USD billions column
df_sbp["it_exports_bn_usd"] = (df_sbp["it_exports_mn_usd"] / 1000).round(3)

out = os.path.join(CLEAN, "sbp_it_exports.csv")
df_sbp.to_csv(out, index=False)

print("✅ SBP IT Export series:")
print(df_sbp.to_string(index=False))
print(f"\n✅ Saved: {out}")

✅ SBP IT Export series:
fiscal_year  it_exports_mn_usd  year  it_exports_yoy_pct  it_exports_bn_usd
       FY06                269  2006                 NaN              0.269
       FY07                227  2007               -15.6              0.227
       FY08                269  2008                18.5              0.269
       FY09                380  2009                41.3              0.380
       FY10                433  2010                13.9              0.433
       FY11                443  2011                 2.3              0.443
       FY12                460  2012                 3.8              0.460
       FY13                803  2013                74.6              0.803
       FY14                817  2014                 1.7              0.817
       FY15                821  2015                 0.5              0.821
       FY16                789  2016                -3.9              0.789
       FY17                940  2017                19.1        

In [39]:
# Source 3: Freelancing Demand Index
# ============================================================
# CELL 5 — Pakistan Freelancing Demand Index
# Sources: Payoneer Global Gig Economy Index, BOI Pakistan,
#          SBP BOP data, PAFLA-ADB report 2024
# ============================================================

freelance_data = {
    "year":                     [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    "freelancers_mn":           [1.0,  1.2,  1.5,  2.0,  2.3,  2.8,  3.0,  3.2,  3.5],
    "freelance_remittances_mn_usd": [150, 180, 220, 280, 320, 396, 397, 420, 460],
    "payoneer_growth_rank":     [None, None, None, 4,    None, 4,   4,   4,   4  ],
    "yoy_revenue_growth_pct":   [None, None, None, None, 20,   47,  2.7, 6,   10 ],
    "avg_hourly_rate_usd":      [None, None, None, None, None, 18,  20,  20,  21 ],
    "female_freelancers_pct":   [None, None, None, None, None, 45,  47,  47,  47 ],
}

df_fl = pd.DataFrame(freelance_data)

# Compute a simple composite "demand index" (0–100 scale)
# normalised on freelancers count × remittances
df_fl["demand_index"] = (
    (df_fl["freelancers_mn"] / df_fl["freelancers_mn"].max()) * 50
    + (df_fl["freelance_remittances_mn_usd"] / df_fl["freelance_remittances_mn_usd"].max()) * 50
).round(1)

out = os.path.join(CLEAN, "freelancing_demand_index.csv")
df_fl.to_csv(out, index=False)

print("✅ Freelancing Demand Index:")
print(df_fl[["year","freelancers_mn","freelance_remittances_mn_usd","demand_index"]].to_string(index=False))
print(f"\n✅ Saved: {out}")

✅ Freelancing Demand Index:
 year  freelancers_mn  freelance_remittances_mn_usd  demand_index
 2016             1.0                           150          30.6
 2017             1.2                           180          36.7
 2018             1.5                           220          45.3
 2019             2.0                           280          59.0
 2020             2.3                           320          67.6
 2021             2.8                           396          83.0
 2022             3.0                           397          86.0
 2023             3.2                           420          91.4
 2024             3.5                           460         100.0

✅ Saved: C:\Users\User\pk_it_demand_dashboard\02_clean_data\freelancing_demand_index.csv


In [40]:
# ============================================================
# CELL 6 (FIXED) — PKR/USD FX rate via fawazahmed0 currency API
# Free, no API key, no rate limits, supports PKR
# Docs: https://github.com/fawazahmed0/exchange-api
# ============================================================

import requests, pandas as pd, os, time, datetime

CLEAN = "../02_clean_data"

def fetch_pkr_rate(date_str):
    """
    Try primary CDN, then fallback CDN.
    date_str format: 'YYYY-MM-DD'
    """
    urls = [
        f"https://cdn.jsdelivr.net/npm/@fawazahmed0/currency-api@{date_str}/v1/currencies/usd.json",
        f"https://{date_str}.currency-api.pages.dev/v1/currencies/usd.json",
    ]
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                rate = data.get("usd", {}).get("pkr")
                if rate:
                    return round(rate, 2)
        except Exception:
            continue
    return None   # both failed → will use fallback

print("📡 Fetching PKR/USD FX rates (2015–2024)...")
print("   (fetching 1st of each month — ~120 requests, ~30 sec)\n")

fx_records = []

for year in range(2015, 2025):
    for month in range(1, 13):
        if datetime.date(year, month, 1) > datetime.date.today():
            continue

        date_str = f"{year}-{month:02d}-01"
        rate = fetch_pkr_rate(date_str)

        if rate:
            fx_records.append({
                "date":        date_str,
                "year":        year,
                "month":       month,
                "pkr_per_usd": rate
            })
        else:
            print(f"  ⚠️  Could not fetch {date_str} — will use annual fallback")

        time.sleep(0.15)

# ── Hard-coded annual averages as guaranteed fallback ──────────────────────
# Source: SBP / Pakistan Bureau of Statistics (official published averages)
annual_fallback = {
    2015: 101.9, 2016: 104.6, 2017: 104.9, 2018: 121.6,
    2019: 150.0, 2020: 161.8, 2021: 167.6, 2022: 204.9,
    2023: 281.0, 2024: 278.5,
}

if len(fx_records) < 10:
    print("\n⚠️  Live fetch returned too few rows — using hard-coded annual averages instead.")
    rows = [{"year": y, "pkr_usd_annual_avg": v} for y, v in annual_fallback.items()]
    df_fx_annual = pd.DataFrame(rows)

    # Expand to monthly (repeat annual avg for all 12 months)
    monthly = []
    for y, avg in annual_fallback.items():
        for m in range(1, 13):
            monthly.append({"date": f"{y}-{m:02d}-01",
                            "year": y, "month": m, "pkr_per_usd": avg})
    df_fx = pd.DataFrame(monthly)

else:
    df_fx = pd.DataFrame(fx_records)
    df_fx_annual = (
        df_fx.groupby("year")["pkr_per_usd"]
        .mean()
        .round(2)
        .reset_index()
        .rename(columns={"pkr_per_usd": "pkr_usd_annual_avg"})
    )

# ── Save ───────────────────────────────────────────────────────────────────
df_fx.to_csv(os.path.join(CLEAN, "fx_pkr_usd_monthly.csv"), index=False)
df_fx_annual.to_csv(os.path.join(CLEAN, "fx_pkr_usd_annual.csv"), index=False)

print(f"\n✅ Monthly FX rows: {len(df_fx)}")
print(f"✅ Annual averages:")
print(df_fx_annual.to_string(index=False))
print(f"\n✅ Saved: fx_pkr_usd_monthly.csv and fx_pkr_usd_annual.csv")

📡 Fetching PKR/USD FX rates (2015–2024)...
   (fetching 1st of each month — ~120 requests, ~30 sec)

  ⚠️  Could not fetch 2015-01-01 — will use annual fallback
  ⚠️  Could not fetch 2015-02-01 — will use annual fallback
  ⚠️  Could not fetch 2015-03-01 — will use annual fallback
  ⚠️  Could not fetch 2015-04-01 — will use annual fallback
  ⚠️  Could not fetch 2015-05-01 — will use annual fallback
  ⚠️  Could not fetch 2015-06-01 — will use annual fallback
  ⚠️  Could not fetch 2015-07-01 — will use annual fallback
  ⚠️  Could not fetch 2015-08-01 — will use annual fallback
  ⚠️  Could not fetch 2015-09-01 — will use annual fallback
  ⚠️  Could not fetch 2015-10-01 — will use annual fallback
  ⚠️  Could not fetch 2015-11-01 — will use annual fallback
  ⚠️  Could not fetch 2015-12-01 — will use annual fallback
  ⚠️  Could not fetch 2016-01-01 — will use annual fallback
  ⚠️  Could not fetch 2016-02-01 — will use annual fallback
  ⚠️  Could not fetch 2016-03-01 — will use annual fallback

OSError: Cannot save file into a non-existent directory: '..\02_clean_data'

In [66]:
# ── Save ───────────────────────────────────────────────────────────────────
import requests, pandas as pd, os, time, datetime

# ── Absolute path: works regardless of which cell set CLEAN before ──
BASE = os.getcwd()
CLEAN = r"C:\Users\User\pk_it_demand_dashboard\02_clean_data"

print(BASE)
print(CLEAN)


os.makedirs(CLEAN, exist_ok=True)   # creates the folder if it doesn't exist
print(f"📁 Saving to: {CLEAN}")

# ── Save ───────────────────────────────────────────────────────────────────
df_fx.to_csv(os.path.join(CLEAN, "fx_pkr_usd_monthly.csv"), index=False)
df_fx_annual.to_csv(os.path.join(CLEAN, "fx_pkr_usd_annual.csv"), index=False)

print(f"\n✅ Monthly FX rows: {len(df_fx)}")
print(f"✅ Annual averages:")
print(df_fx_annual.to_string(index=False))
print(f"\n✅ Saved: fx_pkr_usd_monthly.csv and fx_pkr_usd_annual.csv")

C:\Users\User
C:\Users\User\pk_it_demand_dashboard\02_clean_data
📁 Saving to: C:\Users\User\pk_it_demand_dashboard\02_clean_data

✅ Monthly FX rows: 120
✅ Annual averages:
 year  pkr_usd_annual_avg
 2015               101.9
 2016               104.6
 2017               104.9
 2018               121.6
 2019               150.0
 2020               161.8
 2021               167.6
 2022               204.9
 2023               281.0
 2024               278.5

✅ Saved: fx_pkr_usd_monthly.csv and fx_pkr_usd_annual.csv


In [68]:
# ============================================================
# CELL 7 — Verify all 5 output CSVs
# ============================================================
import os

expected = [
    "wb_macro_indicators.csv",
    "sbp_it_exports.csv",
    "freelancing_demand_index.csv",
    "fx_pkr_usd_monthly.csv",
    "fx_pkr_usd_annual.csv",
]

print("📁 Checking 02_clean_data/\n")
all_ok = True
for fname in expected:
    path = os.path.join(CLEAN, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✅  {fname:45s} → {len(df)} rows × {len(df.columns)} cols")
    else:
        print(f"  ❌  {fname} MISSING")
        all_ok = False

print()
print("🚀 Ready for Phase 3 — Data Cleaning!" if all_ok else "⚠️  Fix missing files above.")


📁 Checking 02_clean_data/

  ✅  wb_macro_indicators.csv                       → 25 rows × 6 cols
  ✅  sbp_it_exports.csv                            → 20 rows × 5 cols
  ✅  freelancing_demand_index.csv                  → 9 rows × 8 cols
  ✅  fx_pkr_usd_monthly.csv                        → 120 rows × 4 cols
  ✅  fx_pkr_usd_annual.csv                         → 10 rows × 2 cols

🚀 Ready for Phase 3 — Data Cleaning!


# Phase 3 — Data Cleaning & Merging

In [71]:
# ============================================================
# CELL 8 — Inspect raw clean files before merging
# ============================================================
import pandas as pd, os, numpy as np

PROJECT_ROOT = r"C:\Users\User\pk_it_demand_dashboard"
CLEAN        = os.path.join(PROJECT_ROOT, "02_clean_data")

files = {
    "wb"  : "wb_macro_indicators.csv",
    "sbp" : "sbp_it_exports.csv",
    "fl"  : "freelancing_demand_index.csv",
    "fx_a": "fx_pkr_usd_annual.csv",
    "fx_m": "fx_pkr_usd_monthly.csv",
}

dfs = {}
for key, fname in files.items():
    df = pd.read_csv(os.path.join(CLEAN, fname))
    dfs[key] = df
    print(f"── {fname}")
    print(f"   Shape  : {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Years  : {sorted(df['year'].unique()) if 'year' in df.columns else 'see date col'}")
    print()

── wb_macro_indicators.csv
   Shape  : (25, 6)
   Columns: ['year', 'gdp_usd', 'internet_users_pct', 'ict_services_exports_pct_service_exports', 'population', 'gdp_usd_bn']
   Years  : [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

── sbp_it_exports.csv
   Shape  : (20, 5)
   Columns: ['fiscal_year', 'it_exports_mn_usd', 'year', 'it_exports_yoy_pct', 'it_exports_bn_usd']
   Years  : [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

── freelancing_demand_index.csv
   Shape  : (9, 8)
   Columns: ['year', 'freelancers_mn', 'freelance_remittances_mn_usd', 'payoneer_growth_rank', 'yoy_revenue_growth_pct', 'avg_hourly_rate_usd', 'female_freelancers_pct', 'demand_index']
   Years  : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

── fx_pkr_usd_annual.csv
   Shape  : (10, 2)
   Columns: ['year', 'pkr_usd_annual_

In [73]:
# ============================================================
# CELL 9 — Clean & standardise all sources
# ============================================================

# ── 1. World Bank ─────────────────────────────────────────
wb = dfs["wb"].copy()
wb = wb[wb["year"] >= 2006].reset_index(drop=True)

# Fill small gaps with linear interpolation
for col in ["internet_users_pct", "ict_services_exports_pct_service_exports",
            "gdp_usd_bn", "population"]:
    if col in wb.columns:
        wb[col] = wb[col].interpolate(method="linear").round(3)

print("✅ World Bank cleaned:")
print(wb[["year","gdp_usd_bn","internet_users_pct"]].to_string(index=False))

# ── 2. SBP IT Exports ─────────────────────────────────────
sbp = dfs["sbp"].copy()
# Keep only the columns we need
sbp = sbp[["year","fiscal_year","it_exports_mn_usd",
           "it_exports_bn_usd","it_exports_yoy_pct"]].copy()

# Label growth periods for Power BI colour coding
def growth_label(pct):
    if pd.isna(pct):   return "Baseline"
    elif pct >= 30:    return "High Growth (30%+)"
    elif pct >= 10:    return "Moderate Growth (10–30%)"
    elif pct >= 0:     return "Slow Growth (0–10%)"
    else:              return "Decline"

sbp["growth_label"] = sbp["it_exports_yoy_pct"].apply(growth_label)
print("\n✅ SBP IT Exports cleaned:")
print(sbp[["year","fiscal_year","it_exports_mn_usd","growth_label"]].to_string(index=False))

# ── 3. Freelancing index ───────────────────────────────────
fl = dfs["fl"].copy()
fl = fl[["year","freelancers_mn","freelance_remittances_mn_usd",
         "demand_index","avg_hourly_rate_usd","female_freelancers_pct"]].copy()

# Interpolate missing values
for col in ["avg_hourly_rate_usd","female_freelancers_pct"]:
    fl[col] = fl[col].interpolate(method="linear").round(1)

print("\n✅ Freelancing index cleaned:")
print(fl.to_string(index=False))

# ── 4. FX annual ──────────────────────────────────────────
fx = dfs["fx_a"].copy()
fx.columns = ["year","pkr_per_usd"]
print("\n✅ FX annual cleaned:")
print(fx.to_string(index=False))

✅ World Bank cleaned:
 year  gdp_usd_bn  internet_users_pct
 2006      161.87               6.500
 2007      184.14               6.800
 2008      202.20               7.000
 2009      187.34               7.500
 2010      196.71               8.000
 2011      230.59               8.000
 2012      250.11               8.100
 2013      258.66               9.000
 2014      271.39              10.000
 2015      299.96              11.000
 2016      313.63              12.385
 2017      339.21              13.780
 2018      356.13              15.340
 2019      320.91              17.071
 2020      300.43              18.935
 2021      348.52              28.514
 2022      374.89              38.094
 2023      336.69              47.674
 2024      371.57              57.253

✅ SBP IT Exports cleaned:
 year fiscal_year  it_exports_mn_usd             growth_label
 2006        FY06                269                 Baseline
 2007        FY07                227                  Decline
 2008

In [75]:
# ============================================================
# CELL 10 — Merge into master_dashboard_data.csv
# ============================================================

# Start from SBP as the spine (FY2006–FY2025, most complete)
master = sbp.copy()

# Merge World Bank
master = master.merge(wb[["year","gdp_usd_bn","internet_users_pct",
                           "ict_services_exports_pct_service_exports",
                           "population"]],
                      on="year", how="left")

# Merge FX
master = master.merge(fx, on="year", how="left")

# Merge freelancing (starts 2016, will be NaN before that — that's fine)
master = master.merge(fl, on="year", how="left")

# ── Derived columns ───────────────────────────────────────

# IT exports in PKR billion (for local currency view)
master["it_exports_pkr_bn"] = (
    master["it_exports_mn_usd"] * master["pkr_per_usd"] / 1000
).round(2)

# IT exports as % of GDP
master["it_exports_pct_gdp"] = (
    (master["it_exports_mn_usd"] / 1000) / master["gdp_usd_bn"] * 100
).round(3)

# Internet penetration tier (for slicer in Power BI)
def internet_tier(pct):
    if pd.isna(pct):    return "Unknown"
    elif pct >= 35:     return "High (35%+)"
    elif pct >= 20:     return "Medium (20–35%)"
    else:               return "Low (<20%)"

master["internet_tier"] = master["internet_users_pct"].apply(internet_tier)

# Decade label
master["decade"] = master["year"].apply(
    lambda y: "2000s" if y < 2010 else ("2010s" if y < 2020 else "2020s")
)

# Sort
master = master.sort_values("year").reset_index(drop=True)

print("✅ Master table shape:", master.shape)
print("\nColumns:", list(master.columns))
print("\nPreview (last 8 rows):")
print(master.tail(8)[["year","fiscal_year","it_exports_mn_usd",
                        "it_exports_yoy_pct","gdp_usd_bn",
                        "pkr_per_usd","demand_index"]].to_string(index=False))

✅ Master table shape: (20, 20)

Columns: ['year', 'fiscal_year', 'it_exports_mn_usd', 'it_exports_bn_usd', 'it_exports_yoy_pct', 'growth_label', 'gdp_usd_bn', 'internet_users_pct', 'ict_services_exports_pct_service_exports', 'population', 'pkr_per_usd', 'freelancers_mn', 'freelance_remittances_mn_usd', 'demand_index', 'avg_hourly_rate_usd', 'female_freelancers_pct', 'it_exports_pkr_bn', 'it_exports_pct_gdp', 'internet_tier', 'decade']

Preview (last 8 rows):
 year fiscal_year  it_exports_mn_usd  it_exports_yoy_pct  gdp_usd_bn  pkr_per_usd  demand_index
 2018        FY18               1067                13.5      356.13        121.6          45.3
 2019        FY19               1192                11.7      320.91        150.0          59.0
 2020        FY20               1440                20.8      300.43        161.8          67.6
 2021        FY21               2108                46.4      348.52        167.6          83.0
 2022        FY22               2619                24.2 

In [77]:
# ============================================================
# CELL 11 — Quality checks + save all files
# ============================================================

print("=" * 55)
print("  DATA QUALITY REPORT")
print("=" * 55)

print(f"\n📐 Master shape : {master.shape[0]} rows × {master.shape[1]} cols")
print(f"📅 Year range   : {master.year.min()} – {master.year.max()}")

print("\n🔍 Null counts per column:")
nulls = master.isnull().sum()
for col, n in nulls.items():
    flag = "⚠️ " if n > 5 else ("·  " if n == 0 else "~  ")
    print(f"   {flag} {col:45s} {n} nulls")

print("\n📊 Key stats:")
print(f"   IT Exports range : ${master.it_exports_mn_usd.min()}M → ${master.it_exports_mn_usd.max()}M")
print(f"   Best YoY growth  : {master.it_exports_yoy_pct.max():.1f}%  ({int(master.loc[master.it_exports_yoy_pct.idxmax(),'year'])})")
print(f"   Worst YoY        : {master.it_exports_yoy_pct.min():.1f}%  ({int(master.loc[master.it_exports_yoy_pct.idxmin(),'year'])})")
print(f"   PKR depreciation : {master[master.year==2015].pkr_per_usd.values[0]} → {master[master.year==2024].pkr_per_usd.values[0]} PKR/USD")

# ── Save master ───────────────────────────────────────────
master_path = os.path.join(CLEAN, "master_dashboard_data.csv")
master.to_csv(master_path, index=False)
print(f"\n✅ Saved: master_dashboard_data.csv")

# ── Save monthly FX separately (for FX trend page in Power BI) ──
fx_monthly = dfs["fx_m"].copy()
fx_monthly["date"] = pd.to_datetime(fx_monthly["date"])
fx_monthly_path = os.path.join(CLEAN, "fx_pkr_usd_monthly.csv")
fx_monthly.to_csv(fx_monthly_path, index=False)
print(f"✅ Saved: fx_pkr_usd_monthly.csv (refreshed)")

# ── Final file checklist ──────────────────────────────────
print("\n📁 Final 02_clean_data contents:")
for f in sorted(os.listdir(CLEAN)):
    size = os.path.getsize(os.path.join(CLEAN, f))
    print(f"   ✅  {f:45s} ({size/1024:.1f} KB)")

print("\n🚀 Phase 3 complete — Ready for Phase 4: Forecasting!")

  DATA QUALITY REPORT

📐 Master shape : 20 rows × 20 cols
📅 Year range   : 2006 – 2025

🔍 Null counts per column:
   ·   year                                          0 nulls
   ·   fiscal_year                                   0 nulls
   ·   it_exports_mn_usd                             0 nulls
   ·   it_exports_bn_usd                             0 nulls
   ~   it_exports_yoy_pct                            1 nulls
   ·   growth_label                                  0 nulls
   ~   gdp_usd_bn                                    1 nulls
   ~   internet_users_pct                            1 nulls
   ~   ict_services_exports_pct_service_exports      1 nulls
   ~   population                                    1 nulls
   ⚠️  pkr_per_usd                                   10 nulls
   ⚠️  freelancers_mn                                11 nulls
   ⚠️  freelance_remittances_mn_usd                  11 nulls
   ⚠️  demand_index                                  11 nulls
   ⚠️  avg_hourly_rate_usd  

# Phase 4 — Forecasting Model (Prophet)

In [80]:
# ============================================================
# CELL 12 — Prophet forecast: Pakistan IT Exports
# ============================================================
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = r"C:\Users\User\pk_it_demand_dashboard"
CLEAN        = os.path.join(PROJECT_ROOT, "02_clean_data")
FORECASTS    = os.path.join(PROJECT_ROOT, "03_forecasts")
os.makedirs(FORECASTS, exist_ok=True)

# ── Load master ───────────────────────────────────────────
master = pd.read_csv(os.path.join(CLEAN, "master_dashboard_data.csv"))

# ── Prophet requires columns: ds (date) and y (value) ────
# We have annual data → set date to July 1 of each fiscal year end
prophet_df = pd.DataFrame({
    "ds": pd.to_datetime(master["year"].astype(str) + "-07-01"),
    "y" : master["it_exports_mn_usd"].astype(float),
})

print("✅ Prophet input shape:", prophet_df.shape)
print(prophet_df.to_string(index=False))

✅ Prophet input shape: (20, 2)
        ds      y
2006-07-01  269.0
2007-07-01  227.0
2008-07-01  269.0
2009-07-01  380.0
2010-07-01  433.0
2011-07-01  443.0
2012-07-01  460.0
2013-07-01  803.0
2014-07-01  817.0
2015-07-01  821.0
2016-07-01  789.0
2017-07-01  940.0
2018-07-01 1067.0
2019-07-01 1192.0
2020-07-01 1440.0
2021-07-01 2108.0
2022-07-01 2619.0
2023-07-01 2596.0
2024-07-01 3223.0
2025-07-01 3809.0


In [82]:
# ============================================================
# CELL 13 — Fit Prophet + forecast 5 years ahead
# ============================================================

# ── Configure Prophet for annual economic data ────────────
model = Prophet(
    growth              = "linear",
    yearly_seasonality  = False,   # annual data — no sub-year seasonality
    weekly_seasonality  = False,
    daily_seasonality   = False,
    changepoint_prior_scale     = 0.3,   # allows meaningful trend shifts
    seasonality_prior_scale     = 10,
    interval_width      = 0.90,          # 90% confidence interval
    n_changepoints      = 8,
)

# Add GDP as an external regressor (macro context)
master_filled = master.copy()
master_filled["gdp_usd_bn"] = master_filled["gdp_usd_bn"].interpolate()
model.add_regressor("gdp_usd_bn", standardize=True)

prophet_df["gdp_usd_bn"] = master_filled["gdp_usd_bn"].values

# ── Fit ───────────────────────────────────────────────────
model.fit(prophet_df)
print("✅ Model fitted on", len(prophet_df), "annual observations")

# ── Build future dataframe (5 years: 2026–2030) ──────────
future_years = pd.DataFrame({
    "ds": pd.to_datetime([f"{y}-07-01" for y in range(2006, 2031)]),
})

# Project GDP forward at 5% annual growth (IMF Pakistan estimate)
last_gdp = master_filled["gdp_usd_bn"].dropna().iloc[-1]
base_year = master_filled["year"].max()
gdp_projection = []
for ds in future_years["ds"]:
    y = ds.year
    if y <= base_year:
        val = master_filled.loc[master_filled["year"] == y, "gdp_usd_bn"]
        gdp_projection.append(val.values[0] if len(val) > 0 else np.nan)
    else:
        gdp_projection.append(round(last_gdp * (1.05 ** (y - base_year)), 2))

future_years["gdp_usd_bn"] = gdp_projection
future_years["gdp_usd_bn"] = pd.to_numeric(
    future_years["gdp_usd_bn"], errors="coerce"
).interpolate()

# ── Predict ───────────────────────────────────────────────
forecast = model.predict(future_years)

# ── Clean up forecast output ──────────────────────────────
forecast_out = forecast[[
    "ds", "yhat", "yhat_lower", "yhat_upper",
    "trend", "trend_lower", "trend_upper"
]].copy()

forecast_out["year"]      = forecast_out["ds"].dt.year
forecast_out["is_actual"] = forecast_out["year"].isin(master["year"].values)

# Clip negative lower bounds (exports can't be negative)
forecast_out["yhat_lower"] = forecast_out["yhat_lower"].clip(lower=0)
forecast_out["yhat"]       = forecast_out["yhat"].round(1)
forecast_out["yhat_lower"] = forecast_out["yhat_lower"].round(1)
forecast_out["yhat_upper"] = forecast_out["yhat_upper"].round(1)

# Add actual values for comparison rows
forecast_out = forecast_out.merge(
    master[["year","it_exports_mn_usd","fiscal_year","it_exports_yoy_pct"]],
    on="year", how="left"
)

# Forecast YoY
forecast_out = forecast_out.sort_values("year").reset_index(drop=True)
forecast_out["forecast_yoy_pct"] = (
    forecast_out["yhat"].pct_change() * 100
).round(1)

# Tag
forecast_out["period_type"] = forecast_out["is_actual"].map(
    {True: "Historical", False: "Forecast"}
)

print("\n✅ Forecast preview (2022 → 2030):")
cols = ["year","fiscal_year","it_exports_mn_usd",
        "yhat","yhat_lower","yhat_upper","period_type"]
print(forecast_out[forecast_out["year"] >= 2022][cols].to_string(index=False))

22:03:37 - cmdstanpy - INFO - Chain [1] start processing
22:03:37 - cmdstanpy - INFO - Chain [1] done processing


✅ Model fitted on 20 annual observations

✅ Forecast preview (2022 → 2030):
 year fiscal_year  it_exports_mn_usd   yhat  yhat_lower  yhat_upper period_type
 2022        FY22             2619.0 2505.5      2358.6      2655.5  Historical
 2023        FY23             2596.0 2786.9      2625.8      2945.0  Historical
 2024        FY24             3223.0 3284.4      3133.9      3440.4  Historical
 2025        FY25             3809.0 3678.3      3527.4      3837.5  Historical
 2026         NaN                NaN 4126.8      3968.6      4280.0    Forecast
 2027         NaN                NaN 4578.0      4397.4      4756.8    Forecast
 2028         NaN                NaN 5033.2      4757.9      5297.5    Forecast
 2029         NaN                NaN 5490.3      5102.9      5853.1    Forecast
 2030         NaN                NaN 5950.6      5410.2      6421.7    Forecast


In [84]:
# ============================================================
# CELL 14 — Trend components + model accuracy
# ============================================================
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# ── Trend components (for decomposition page in Power BI) ─
components = forecast[["ds","trend","trend_lower","trend_upper"]].copy()
components["year"] = components["ds"].dt.year
components = components.merge(
    master[["year","it_exports_mn_usd"]], on="year", how="left"
)

# ── Accuracy on historical period ─────────────────────────
hist = forecast_out[forecast_out["is_actual"]].copy()
hist = hist.dropna(subset=["it_exports_mn_usd"])

mae  = mean_absolute_error(hist["it_exports_mn_usd"], hist["yhat"])
mape = mean_absolute_percentage_error(hist["it_exports_mn_usd"], hist["yhat"]) * 100
rmse = np.sqrt(((hist["it_exports_mn_usd"] - hist["yhat"]) ** 2).mean())

metrics = pd.DataFrame([{
    "metric"       : "MAE (Mean Absolute Error)",
    "value"        : round(mae, 1),
    "unit"         : "Million USD",
    "interpretation": f"Forecast is off by ~${mae:.0f}M on average"
},{
    "metric"       : "MAPE (Mean Absolute % Error)",
    "value"        : round(mape, 1),
    "unit"         : "%",
    "interpretation": f"{'Excellent' if mape<10 else 'Good' if mape<20 else 'Acceptable'} fit ({mape:.1f}%)"
},{
    "metric"       : "RMSE",
    "value"        : round(rmse, 1),
    "unit"         : "Million USD",
    "interpretation": f"Root mean squared error: ${rmse:.0f}M"
},{
    "metric"       : "Training Observations",
    "value"        : len(hist),
    "unit"         : "years",
    "interpretation": f"Model trained on {len(hist)} annual data points"
}])

print("📊 Model Accuracy Metrics:")
print(metrics[["metric","value","unit","interpretation"]].to_string(index=False))

# ── Save all 3 forecast files ──────────────────────────────
forecast_out.to_csv(os.path.join(FORECASTS, "forecast_it_exports.csv"), index=False)
components.to_csv(os.path.join(FORECASTS,   "trend_components.csv"),    index=False)
metrics.to_csv(os.path.join(FORECASTS,      "model_accuracy.csv"),      index=False)

print(f"\n✅ Saved: forecast_it_exports.csv")
print(f"✅ Saved: trend_components.csv")
print(f"✅ Saved: model_accuracy.csv")

print("\n📁 03_forecasts contents:")
for f in sorted(os.listdir(FORECASTS)):
    size = os.path.getsize(os.path.join(FORECASTS, f))
    print(f"   ✅  {f:40s} ({size/1024:.1f} KB)")

print("\n🎉 Phase 4 complete — Day 1 DONE!")
print("   Tomorrow: Power BI data model + 4-page dashboard")

📊 Model Accuracy Metrics:
                      metric  value        unit                         interpretation
   MAE (Mean Absolute Error)   81.0 Million USD    Forecast is off by ~$81M on average
MAPE (Mean Absolute % Error)   11.6           %                       Good fit (11.6%)
                        RMSE   93.0 Million USD          Root mean squared error: $93M
       Training Observations   20.0       years Model trained on 20 annual data points

✅ Saved: forecast_it_exports.csv
✅ Saved: trend_components.csv
✅ Saved: model_accuracy.csv

📁 03_forecasts contents:
   ✅  forecast_it_exports.csv                  (3.2 KB)
   ✅  model_accuracy.csv                       (0.3 KB)
   ✅  trend_components.csv                     (1.9 KB)

🎉 Phase 4 complete — Day 1 DONE!
   Tomorrow: Power BI data model + 4-page dashboard
